# Managerial Insights from the Baseline Hybrid Rollout Run

This notebook supports the managerial-insight chapter of the thesis. The primary evidence is the resulting hybrid rollout run using the `EX15_FleetBrokenGlobal` Linear VFA configuration. The notebook therefore focuses on explaining the behavior of the integrated policy: when it rebalances, when it performs maintenance, how these decisions vary over time, and what operational implications follow.

Comparative experiments are still useful, but they are treated as optional extensions. Where a managerial question cannot be answered causally from the single rollout run, the notebook states what additional counterfactual simulation output would be required.

## Primary Data Source

The default run folder is:

`run_logs/baseline_maintenanceextension_run_logs/hybrid_EX15_FleetBrokenGlobal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1`

Expected files:

- `results.csv`: one row per seed with final service level and aggregate outcomes.
- `seed_*/hourly_metrics.csv`: hourly service, breakdown, restoration, and damaged-fleet metrics.
- `seed_*/daily_metrics.csv`: daily aggregate outcomes.
- `seed_*/decisions.csv`: decision-level action composition and vehicle state.

In [ ]:
from pathlib import Path
import os
import re
import warnings

# Keep Matplotlib cache local to the project when running on restricted machines.
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PRIMARY_RUN_DIR = Path(
    "run_logs/baseline_maintenanceextension_run_logs/"
    "hybrid_EX15_FleetBrokenGlobal_seed1000_bias_term_trans100_fsdiag25_g0.97_eps0_initbm2p5_TD_W34_old_D504h_V1"
)

# Add folders here only when you have explicit counterfactual runs.
OPTIONAL_COMPARISON_RUN_DIRS = []

POLICY_LABEL = "Integrated hybrid rollout (EX15 FleetBrokenGlobal)"
PEAK_HOURS = [(7, 9), (15, 18)]
OUTPUT_DIR = Path("managerial_insights_figures")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# Loading utilities

def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        warnings.warn(f"Missing file: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def load_run_folder(run_dir: Path, policy_label: str) -> dict[str, pd.DataFrame]:
    run_dir = Path(run_dir)
    results = read_csv_if_exists(run_dir / "results.csv")
    if not results.empty:
        results["source_file"] = str(run_dir / "results.csv")
        results["policy_label"] = policy_label
        if "policy_name" not in results.columns:
            results["policy_name"] = results.get("exp_name", policy_label)

    frames = {"hourly": [], "daily": [], "decisions": []}
    for seed_dir in sorted(run_dir.glob("seed_*")):
        seed_match = re.search(r"seed_(\d+)", seed_dir.name)
        seed_value = int(seed_match.group(1)) if seed_match else np.nan
        for kind, filename in {
            "hourly": "hourly_metrics.csv",
            "daily": "daily_metrics.csv",
            "decisions": "decisions.csv",
        }.items():
            df = read_csv_if_exists(seed_dir / filename)
            if df.empty:
                continue
            if "seed" not in df.columns:
                df["seed"] = seed_value
            df["source_file"] = str(seed_dir / filename)
            df["policy_label"] = policy_label
            frames[kind].append(df)

    return {
        "results": results,
        "hourly": pd.concat(frames["hourly"], ignore_index=True, sort=False) if frames["hourly"] else pd.DataFrame(),
        "daily": pd.concat(frames["daily"], ignore_index=True, sort=False) if frames["daily"] else pd.DataFrame(),
        "decisions": pd.concat(frames["decisions"], ignore_index=True, sort=False) if frames["decisions"] else pd.DataFrame(),
    }


def load_optional_comparisons(run_dirs):
    loaded = []
    for run_dir in run_dirs:
        label = Path(run_dir).name
        loaded.append(load_run_folder(Path(run_dir), label))
    if not loaded:
        return {"results": pd.DataFrame(), "hourly": pd.DataFrame(), "daily": pd.DataFrame(), "decisions": pd.DataFrame()}
    return {
        kind: pd.concat([item[kind] for item in loaded if not item[kind].empty], ignore_index=True, sort=False)
        for kind in ["results", "hourly", "daily", "decisions"]
    }


primary = load_run_folder(PRIMARY_RUN_DIR, POLICY_LABEL)
comparisons = load_optional_comparisons(OPTIONAL_COMPARISON_RUN_DIRS)

results = primary["results"]
hourly = primary["hourly"]
daily = primary["daily"]
decisions = primary["decisions"]

for name, df in primary.items():
    print(f"primary {name:9s}: {df.shape[0]:>8,} rows, {df.shape[1]:>3} columns")

In [ ]:
# Cleaning and derived variables

def numeric(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def add_time_features(df):
    df = df.copy()
    if "hour" in df.columns:
        df["hour"] = pd.to_numeric(df["hour"], errors="coerce") % 24
    if "day" in df.columns:
        df["day"] = pd.to_numeric(df["day"], errors="coerce")
    if {"day", "hour"}.issubset(df.columns):
        df["absolute_hour"] = df["day"] * 24 + df["hour"]
    if {"day", "hour", "minute"}.issubset(df.columns):
        df["time_min"] = df["day"] * 24 * 60 + df["hour"] * 60 + pd.to_numeric(df["minute"], errors="coerce")
    return df


def is_peak_hour(hour):
    if pd.isna(hour):
        return False
    hour = int(hour)
    return any(start <= hour < end for start, end in PEAK_HOURS)


def add_service_metrics(df, prefix=""):
    df = numeric(df, ["service_level", "starvations", "congestions", "total_trips", "daily_starvations", "daily_congestions", "daily_trips"])
    if {"starvations", "congestions", "total_trips"}.issubset(df.columns):
        failures = df["starvations"].fillna(0) + df["congestions"].fillna(0)
        df["computed_service_level"] = 1 - failures / df["total_trips"].replace(0, np.nan)
    if {"daily_starvations", "daily_congestions", "daily_trips"}.issubset(df.columns):
        failures = df["daily_starvations"].fillna(0) + df["daily_congestions"].fillna(0)
        df["daily_service_level"] = 1 - failures / df["daily_trips"].replace(0, np.nan)
    return df


def add_action_metrics(df):
    df = numeric(df, [
        "functional_pickups", "functional_deliveries", "onsite_repairs", "depot_pickups",
        "depot_deliveries", "load_from_queue", "action_duration_min", "travel_time_min",
        "functional_load_before", "depot_load_before", "total_load_before",
        "functional_load_after", "depot_load_after", "total_load_after",
    ])
    for col in ["functional_pickups", "functional_deliveries", "onsite_repairs", "depot_pickups", "depot_deliveries", "load_from_queue"]:
        if col not in df.columns:
            df[col] = 0
    is_at_depot = df.get("is_at_depot", pd.Series(False, index=df.index)).astype(str).str.lower().isin(["true", "1", "yes"])
    next_is_depot = df.get("next_station_id", pd.Series("", index=df.index)).astype(str).str.startswith("D")
    selected_maintenance = df.get("selected_action_is_maintenance", pd.Series(False, index=df.index)).astype(str).str.lower().isin(["true", "1", "yes"])

    df["is_at_depot_bool"] = is_at_depot
    df["next_is_depot"] = next_is_depot
    df["depot_return_decision"] = next_is_depot & ~is_at_depot
    df["maintenance_action"] = selected_maintenance | (df["onsite_repairs"] > 0) | (df["depot_pickups"] > 0) | (df["depot_deliveries"] > 0) | (df["load_from_queue"] > 0) | is_at_depot
    df["rebalancing_action"] = (df["functional_pickups"] > 0) | (df["functional_deliveries"] > 0)
    df["mixed_action"] = df["maintenance_action"] & df["rebalancing_action"]
    df["period"] = np.where(df.get("hour", pd.Series(np.nan, index=df.index)).apply(is_peak_hour), "Peak", "Off-peak")

    conditions = [
        df["depot_return_decision"],
        is_at_depot & ((df["depot_deliveries"] > 0) | (df["load_from_queue"] > 0)),
        df["onsite_repairs"] > 0,
        df["depot_pickups"] > 0,
        df["mixed_action"],
        df["rebalancing_action"],
        is_at_depot,
    ]
    labels = ["go_to_depot", "depot_service", "onsite_repair", "depot_collection", "mixed", "rebalancing", "depot_passthrough"]
    df["derived_action_type"] = np.select(conditions, labels, default="no_operation")

    if "winning_profile_type" in df.columns:
        df["profile_type"] = df["winning_profile_type"].replace("", np.nan).fillna(df["derived_action_type"])
    else:
        df["profile_type"] = df["derived_action_type"]

    load_cols = [c for c in ["total_load_before", "total_load_after"] if c in df.columns]
    if load_cols:
        df["estimated_vehicle_capacity"] = df.groupby("seed")[load_cols].transform("max").max(axis=1)
    else:
        df["estimated_vehicle_capacity"] = np.nan
    df["broken_load_before"] = df.get("depot_load_before", np.nan)
    df["broken_load_after"] = df.get("depot_load_after", df["broken_load_before"])
    df["broken_load_to_depot"] = np.where(df["depot_return_decision"], df["broken_load_after"], np.nan)
    df["broken_load_utilization_to_depot"] = df["broken_load_to_depot"] / df["estimated_vehicle_capacity"].replace(0, np.nan)
    return df


results = add_service_metrics(results)
hourly = add_service_metrics(add_time_features(hourly))
daily = add_service_metrics(add_time_features(daily))
decisions = add_action_metrics(add_time_features(decisions))

if not hourly.empty:
    hourly["period"] = np.where(hourly["hour"].apply(is_peak_hour), "Peak", "Off-peak")
    hourly["damaged_fraction_total"] = hourly.get("damaged_fraction_onsite", 0).fillna(0) + hourly.get("damaged_fraction_depot", 0).fillna(0)
if not daily.empty:
    daily["eod_damaged_fraction_total"] = daily.get("eod_damaged_fraction_onsite", 0).fillna(0) + daily.get("eod_damaged_fraction_depot", 0).fillna(0)

In [ ]:
# Plot and summary helpers

def mean_ci(df, group_cols, value_col):
    if df.empty or value_col not in df.columns:
        return pd.DataFrame()
    out = df.dropna(subset=[value_col]).groupby(group_cols, dropna=False)[value_col].agg(mean="mean", std="std", n="count").reset_index()
    out["se"] = out["std"] / np.sqrt(out["n"].clip(lower=1))
    out["ci95"] = 1.96 * out["se"]
    return out


def add_ci_errorbar(ax, x, y, yerr, **kwargs):
    ax.errorbar(x, y, yerr=yerr, marker="o", capsize=4, **kwargs)


def savefig(name):
    path = OUTPUT_DIR / name
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved {path}")


def show_required(message):
    print(f"Additional simulation output required: {message}")

## Baseline Run Overview

This section summarizes the primary rollout run before moving into the managerial topics. The baseline contains multiple evaluation seeds, so the main uncertainty shown in the figures is across seeds rather than across alternative policies.

In [ ]:
# Final outcome summary across seeds.
if not results.empty:
    outcome_cols = [
        "service_level", "starvations", "congestions", "total_trips",
        "total_fixed_on_site", "total_picked_up_to_depot", "total_depot_visits",
        "total_functional_pickups", "total_functional_deliveries",
        "broken_ratio_end_onsite", "broken_ratio_end_depot", "functional_ratio_end",
    ]
    available = [c for c in outcome_cols if c in results.columns]
    summary = results[available].agg(["mean", "std", "min", "max"]).T
    display(summary.round(4))

    fig, ax = plt.subplots()
    sns.histplot(data=results, x="service_level", bins=8, ax=ax)
    ax.axvline(results["service_level"].mean(), color="black", linestyle="--", label=f"Mean = {results['service_level'].mean():.3f}")
    ax.set_title("Distribution of Service Level Across Seeds")
    ax.set_xlabel("Service level")
    ax.legend()
    plt.tight_layout()
else:
    print("No results.csv loaded.")

In [ ]:
# Hourly system performance over the evaluation horizon.
if not hourly.empty:
    hourly_by_clock = mean_ci(hourly, ["hour"], "computed_service_level")
    damage_by_clock = mean_ci(hourly, ["hour"], "damaged_fraction_total")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    add_ci_errorbar(axes[0], hourly_by_clock["hour"], hourly_by_clock["mean"], hourly_by_clock["ci95"])
    axes[0].set_title("Average Service Level by Hour of Day")
    axes[0].set_xlabel("Hour")
    axes[0].set_ylabel("Computed hourly service level")

    add_ci_errorbar(axes[1], damage_by_clock["hour"], damage_by_clock["mean"], damage_by_clock["ci95"], color="tab:red")
    axes[1].set_title("Damaged-Fleet Fraction by Hour of Day")
    axes[1].set_xlabel("Hour")
    axes[1].set_ylabel("On-site + depot damaged fraction")
    plt.tight_layout()
else:
    print("No hourly_metrics.csv files loaded.")

## 1. Systematic Impact of Decoupled Operations

**Managerial question.** What operational value appears to come from integrating rebalancing and maintenance decisions in one rollout policy?

With only the baseline integrated run, the analysis cannot estimate the full integration gain against night-only or hybrid restricted policies. Instead, it documents how often the integrated policy actually uses its flexibility: whether maintenance and rebalancing are interleaved during the day, whether mixed decisions occur, and whether maintenance is selected when opportunities exist.

In [ ]:
# Action composition of the integrated rollout policy.
if not decisions.empty:
    action_counts = decisions["derived_action_type"].value_counts().rename_axis("action_type").reset_index(name="count")
    action_counts["share"] = action_counts["count"] / action_counts["count"].sum()
    display(action_counts)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    sns.barplot(data=action_counts, x="action_type", y="share", ax=axes[0])
    axes[0].set_title("Integrated Policy Action Composition")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Share of decisions")
    axes[0].tick_params(axis="x", rotation=35)

    mix_summary = pd.DataFrame({
        "category": ["Maintenance", "Rebalancing", "Mixed maintenance and rebalancing"],
        "share": [decisions["maintenance_action"].mean(), decisions["rebalancing_action"].mean(), decisions["mixed_action"].mean()],
    })
    sns.barplot(data=mix_summary, x="category", y="share", ax=axes[1])
    axes[1].set_title("Use of Integrated Flexibility")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Share of decisions")
    axes[1].tick_params(axis="x", rotation=25)
    plt.tight_layout()
else:
    print("No decisions.csv files loaded.")

In [ ]:
# Maintenance opportunity versus selected maintenance.
if not decisions.empty and "maintenance_flag_present" in decisions.columns:
    opportunity = decisions.copy()
    opportunity["maintenance_opportunity"] = opportunity["maintenance_flag_present"].astype(str).str.lower().isin(["true", "1", "yes"])
    opp_summary = opportunity.groupby(["maintenance_opportunity", "period"], dropna=False).agg(
        n_decisions=("derived_action_type", "size"),
        selected_maintenance_share=("maintenance_action", "mean"),
        selected_rebalancing_share=("rebalancing_action", "mean"),
    ).reset_index()
    display(opp_summary)

    fig, ax = plt.subplots()
    sns.barplot(data=opp_summary, x="period", y="selected_maintenance_share", hue="maintenance_opportunity", ax=ax)
    ax.set_title("Maintenance Selection Conditional on Maintenance Opportunity")
    ax.set_ylabel("Share of decisions selecting maintenance")
    ax.set_xlabel("")
    plt.tight_layout()
else:
    show_required("`maintenance_flag_present` in decisions.csv to distinguish maintenance opportunity from selected maintenance.")

**Interpretation.** This evidence shows whether integration is actively used by the policy, not merely allowed by the model. If maintenance, rebalancing, and mixed actions all occur in meaningful shares, the integrated formulation is operationally relevant. A direct integration-gain estimate still requires counterfactual runs where maintenance is restricted to night periods or separated from daytime rebalancing.

**Counterfactual runs needed for causal integration gain.** Re-run the same trained VFA with candidate-set restrictions: night maintenance only, hybrid day/night 1, hybrid day/night 2, and unrestricted integrated. Log the active restriction and the removed candidate types at each decision.

## 2. Temporal Sensitivity Study

**Managerial question.** Does the integrated policy naturally shift maintenance away from peak hours, or does it perform maintenance opportunistically throughout the day?

The single rollout run can answer this behaviorally by comparing peak and off-peak action composition, service levels, and broken-bike accumulation.

In [ ]:
if not decisions.empty:
    hour_action = decisions.groupby(["hour", "derived_action_type"]).size().reset_index(name="count")
    hour_total = decisions.groupby("hour").size().rename("total").reset_index()
    hour_action = hour_action.merge(hour_total, on="hour", how="left")
    hour_action["share"] = hour_action["count"] / hour_action["total"]

    fig, ax = plt.subplots(figsize=(11, 5))
    sns.lineplot(data=hour_action, x="hour", y="share", hue="derived_action_type", marker="o", ax=ax)
    ax.set_title("Action Composition by Hour of Day")
    ax.set_ylabel("Share of hourly decisions")
    ax.set_xlabel("Hour")
    plt.tight_layout()

    peak_mix = decisions.groupby(["period", "seed"], dropna=False).agg(
        maintenance_share=("maintenance_action", "mean"),
        rebalancing_share=("rebalancing_action", "mean"),
        onsite_repairs=("onsite_repairs", "sum"),
        depot_pickups=("depot_pickups", "sum"),
        depot_returns=("depot_return_decision", "sum"),
    ).reset_index()
    display(peak_mix.groupby("period").mean(numeric_only=True).round(4))

    fig, ax = plt.subplots()
    peak_long = peak_mix.melt(id_vars=["period", "seed"], value_vars=["maintenance_share", "rebalancing_share"], var_name="decision_share", value_name="share")
    sns.barplot(data=peak_long, x="period", y="share", hue="decision_share", errorbar="se", ax=ax)
    ax.set_title("Peak Versus Off-Peak Decision Allocation")
    ax.set_ylabel("Share of decisions")
    ax.set_xlabel("")
    plt.tight_layout()
else:
    print("No decisions.csv files loaded.")

In [ ]:
if not hourly.empty:
    temporal_summary = hourly.groupby(["period", "seed"], dropna=False).agg(
        service_level=("computed_service_level", "mean"),
        damaged_fraction=("damaged_fraction_total", "mean"),
        starvations=("starvations", "sum"),
        congestions=("congestions", "sum"),
        trips=("total_trips", "sum"),
    ).reset_index()
    display(temporal_summary.groupby("period").mean(numeric_only=True).round(4))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.barplot(data=temporal_summary, x="period", y="service_level", errorbar="se", ax=axes[0])
    axes[0].set_title("Service Level: Peak Versus Off-Peak")
    axes[0].set_ylabel("Hourly service level")
    axes[0].set_xlabel("")

    sns.barplot(data=temporal_summary, x="period", y="damaged_fraction", errorbar="se", ax=axes[1])
    axes[1].set_title("Damaged-Fleet Fraction: Peak Versus Off-Peak")
    axes[1].set_ylabel("Damaged fleet fraction")
    axes[1].set_xlabel("")
    plt.tight_layout()
else:
    print("No hourly_metrics.csv files loaded.")

**Interpretation.** If maintenance shares are lower during peak periods without imposing explicit restrictions, the rollout policy appears to internalize the opportunity cost of maintenance during high-demand periods. If maintenance remains frequent during peaks while service level is stable, this supports full operational flexibility. If service level deteriorates during peaks while maintenance remains high, a peak-hour restriction is a plausible counterfactual to test.

## 3. Viability Thresholds for On-Site Repairs

**Managerial question.** When is on-site repair operationally worthwhile compared with transporting broken bikes to the depot?

The baseline run does not vary the repair-time parameter, so it cannot identify a causal break-even threshold. It can still show the realized balance between on-site repairs and depot collection under the chosen repair-time setting.

In [ ]:
if not decisions.empty:
    repair_mode = decisions.groupby("seed", dropna=False).agg(
        onsite_repairs=("onsite_repairs", "sum"),
        depot_pickups=("depot_pickups", "sum"),
        depot_returns=("depot_return_decision", "sum"),
        maintenance_decisions=("maintenance_action", "sum"),
        total_decisions=("derived_action_type", "size"),
    ).reset_index()
    repair_mode["onsite_share_of_repairs"] = repair_mode["onsite_repairs"] / (repair_mode["onsite_repairs"] + repair_mode["depot_pickups"]).replace(0, np.nan)
    display(repair_mode.describe().T.round(4))

    mode_totals = repair_mode[["onsite_repairs", "depot_pickups", "depot_returns"]].sum().rename_axis("mode").reset_index(name="count")
    fig, ax = plt.subplots()
    sns.barplot(data=mode_totals, x="mode", y="count", ax=ax)
    ax.set_title("Realized Maintenance Mode in the Baseline Run")
    ax.set_xlabel("")
    ax.set_ylabel("Total count across seeds")
    plt.tight_layout()

    duration_by_type = decisions.loc[decisions["derived_action_type"].isin(["onsite_repair", "depot_collection", "rebalancing", "mixed"])]
    if not duration_by_type.empty and "action_duration_min" in duration_by_type.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.boxplot(data=duration_by_type, x="derived_action_type", y="action_duration_min", ax=ax)
        ax.set_title("Stationary Action Duration by Selected Action Type")
        ax.set_xlabel("")
        ax.set_ylabel("Action duration (minutes, excluding travel)")
        ax.tick_params(axis="x", rotation=25)
        plt.tight_layout()
else:
    print("No decisions.csv files loaded.")

**Interpretation.** The realized on-site repair share describes whether the integrated policy considered immediate repair attractive under the current repair-time assumption. A break-even rule such as “perform on-site repairs only below X minutes” requires additional runs with different `tau_rep` values while keeping all other parameters fixed.

**Additional experiment.** Run the same trained or retrained policy for `tau_rep_values = [1, 3, 5, 7, 10, 15]`, log `repair_time`, and compare service level, on-site repairs, depot pickups, and vehicle time spent stationary.

## 4. Optimal Balance Between Maintenance and Rebalancing

**Managerial question.** Does the policy behave like a fixed maintenance/rebalancing split, or does it dynamically shift effort with system conditions?

The baseline run can address this by relating daily and hourly action composition to service level, demand volume, and damaged-bike state.

In [ ]:
if not decisions.empty and not daily.empty:
    daily_actions = decisions.groupby(["seed", "day"], dropna=False).agg(
        maintenance_share=("maintenance_action", "mean"),
        rebalancing_share=("rebalancing_action", "mean"),
        mixed_share=("mixed_action", "mean"),
        onsite_repairs=("onsite_repairs", "sum"),
        depot_pickups=("depot_pickups", "sum"),
        functional_pickups=("functional_pickups", "sum"),
        functional_deliveries=("functional_deliveries", "sum"),
    ).reset_index()
    daily_joined = daily.merge(daily_actions, on=["seed", "day"], how="left")
    display(daily_joined[["seed", "day", "daily_service_level", "maintenance_share", "rebalancing_share", "eod_damaged_fraction_total"]].head())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.scatterplot(data=daily_joined, x="maintenance_share", y="daily_service_level", hue="eod_damaged_fraction_total", palette="viridis", ax=axes[0])
    axes[0].set_title("Daily Service Level Versus Maintenance Share")
    axes[0].set_xlabel("Maintenance share of decisions")
    axes[0].set_ylabel("Daily service level")

    sns.scatterplot(data=daily_joined, x="rebalancing_share", y="daily_service_level", hue="eod_damaged_fraction_total", palette="viridis", ax=axes[1], legend=False)
    axes[1].set_title("Daily Service Level Versus Rebalancing Share")
    axes[1].set_xlabel("Rebalancing share of decisions")
    axes[1].set_ylabel("Daily service level")
    plt.tight_layout()
else:
    show_required("both daily_metrics.csv and decisions.csv to relate daily action composition to daily service outcomes.")

In [ ]:
if not decisions.empty and not hourly.empty:
    hourly_demand = hourly.groupby(["seed", "day", "hour"], dropna=False).agg(
        trips=("total_trips", "sum"),
        service_level=("computed_service_level", "mean"),
        damaged_fraction=("damaged_fraction_total", "mean"),
    ).reset_index()
    hourly_actions = decisions.groupby(["seed", "day", "hour"], dropna=False).agg(
        maintenance_share=("maintenance_action", "mean"),
        rebalancing_share=("rebalancing_action", "mean"),
        n_decisions=("derived_action_type", "size"),
    ).reset_index()
    hourly_joined = hourly_demand.merge(hourly_actions, on=["seed", "day", "hour"], how="left")
    hourly_joined = hourly_joined.dropna(subset=["trips", "maintenance_share"])
    hourly_joined["demand_level"] = pd.qcut(hourly_joined["trips"].rank(method="first"), q=3, labels=["Low", "Medium", "High"])

    demand_mix = hourly_joined.groupby(["demand_level", "seed"], observed=False).agg(
        maintenance_share=("maintenance_share", "mean"),
        rebalancing_share=("rebalancing_share", "mean"),
        service_level=("service_level", "mean"),
    ).reset_index()
    display(demand_mix.groupby("demand_level", observed=False).mean(numeric_only=True).round(4))

    fig, ax = plt.subplots()
    long = demand_mix.melt(id_vars=["demand_level", "seed"], value_vars=["maintenance_share", "rebalancing_share"], var_name="decision_type", value_name="share")
    sns.barplot(data=long, x="demand_level", y="share", hue="decision_type", errorbar="se", ax=ax)
    ax.set_title("Decision Balance by Demand Level")
    ax.set_xlabel("Demand level based on hourly trips")
    ax.set_ylabel("Share of decisions")
    plt.tight_layout()
else:
    show_required("hourly demand and decision logs to analyze dynamic action balance.")

**Interpretation.** If the maintenance share changes with demand level, hour, or damaged-fleet state, a fixed operational split is unlikely to be sufficient. A dynamic policy is especially valuable when high maintenance effort appears on days or hours with elevated damaged-bike pressure rather than uniformly across the horizon.

## 5. Marginal Benefit of Increased Service Vehicle Capacity

**Managerial question.** Would additional service vehicles materially improve performance?

The baseline run uses one service vehicle (`V1` in the folder name), so it cannot estimate marginal returns to adding vehicles. It can, however, show whether the single vehicle appears heavily utilized and where bottlenecks occur.

In [ ]:
if not decisions.empty:
    utilization_hour = decisions.groupby(["seed", "hour"], dropna=False).agg(
        decisions=("derived_action_type", "size"),
        operation_minutes=("action_duration_min", "sum"),
        travel_minutes=("travel_time_min", "sum"),
        maintenance_actions=("maintenance_action", "sum"),
        rebalancing_actions=("rebalancing_action", "sum"),
    ).reset_index()
    utilization_hour["active_minutes_proxy"] = utilization_hour["operation_minutes"].fillna(0) + utilization_hour["travel_minutes"].fillna(0)

    hour_util = mean_ci(utilization_hour, ["hour"], "active_minutes_proxy")
    fig, ax = plt.subplots()
    add_ci_errorbar(ax, hour_util["hour"], hour_util["mean"], hour_util["ci95"])
    ax.set_title("Single-Vehicle Active-Time Proxy by Hour")
    ax.set_xlabel("Hour")
    ax.set_ylabel("Operation + travel minutes per seed-hour")
    plt.tight_layout()

    display(utilization_hour[["operation_minutes", "travel_minutes", "active_minutes_proxy", "maintenance_actions", "rebalancing_actions"]].describe().T.round(3))
else:
    print("No decisions.csv files loaded.")

**Interpretation.** High active-time proxy during many hours suggests that the single vehicle is a potential capacity bottleneck. This is only indirect evidence. A true marginal benefit curve requires runs with `num_service_vehicles_values = [1, 2, 3, 4]` under the same demand, maintenance, and VFA settings.

**Additional experiment.** For each vehicle count, log service level, hourly failures, action composition, vehicle-level active/idle/travel/repair time, and damaged-bike accumulation.

## 6. Depot Strategy and Optimal Vehicle Filling Level

**Managerial question.** Does the policy wait until the vehicle is full of broken bikes before returning to the depot, or does it return earlier when operationally useful?

The decision log can answer this directly by examining the broken-bike load when the selected next station is the depot.

In [ ]:
if not decisions.empty:
    depot_returns = decisions[decisions["depot_return_decision"]].copy()
    if not depot_returns.empty:
        depot_summary = depot_returns.groupby("seed", dropna=False).agg(
            depot_returns=("depot_return_decision", "sum"),
            avg_broken_load=("broken_load_to_depot", "mean"),
            avg_load_utilization=("broken_load_utilization_to_depot", "mean"),
            median_load_utilization=("broken_load_utilization_to_depot", "median"),
        ).reset_index()
        display(depot_summary.describe().T.round(4))

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(data=depot_returns, x="broken_load_to_depot", bins=range(0, int(depot_returns["estimated_vehicle_capacity"].max()) + 2), ax=axes[0])
        axes[0].set_title("Broken-Bike Load When Returning to Depot")
        axes[0].set_xlabel("Broken bikes carried after station action")

        sns.histplot(data=depot_returns, x="broken_load_utilization_to_depot", bins=12, ax=axes[1])
        axes[1].set_title("Vehicle Fill Level at Depot Return")
        axes[1].set_xlabel("Broken load / estimated vehicle capacity")
        plt.tight_layout()

        fig, ax = plt.subplots()
        sns.boxplot(data=depot_returns, x="hour", y="broken_load_to_depot", ax=ax)
        ax.set_title("Depot Return Load by Hour of Day")
        ax.set_xlabel("Hour")
        ax.set_ylabel("Broken bikes carried")
        plt.tight_layout()
    else:
        print("No decisions where next_station_id is a depot were found.")
else:
    print("No decisions.csv files loaded.")

In [ ]:
# Relate depot-return behavior to daily outcomes.
if not decisions.empty and not daily.empty:
    depot_daily = decisions.groupby(["seed", "day"], dropna=False).agg(
        depot_returns=("depot_return_decision", "sum"),
        avg_depot_return_load=("broken_load_to_depot", "mean"),
        avg_depot_return_utilization=("broken_load_utilization_to_depot", "mean"),
    ).reset_index()
    depot_daily = daily.merge(depot_daily, on=["seed", "day"], how="left")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.scatterplot(data=depot_daily, x="depot_returns", y="daily_service_level", hue="eod_damaged_fraction_total", palette="viridis", ax=axes[0])
    axes[0].set_title("Daily Service Level Versus Depot Return Frequency")
    axes[0].set_xlabel("Depot returns per day")
    axes[0].set_ylabel("Daily service level")

    sns.scatterplot(data=depot_daily, x="avg_depot_return_utilization", y="daily_service_level", hue="eod_damaged_fraction_total", palette="viridis", ax=axes[1], legend=False)
    axes[1].set_title("Daily Service Level Versus Depot Return Fill Level")
    axes[1].set_xlabel("Average depot-return utilization")
    axes[1].set_ylabel("Daily service level")
    plt.tight_layout()
else:
    show_required("daily metrics and decision-level depot-return load to relate depot behavior to outcomes.")

**Interpretation.** If depot returns often occur below full capacity, the policy is not simply following a fill-the-vehicle rule. Early returns may be rational when remaining shift time, route position, or depot repair queues make unloading valuable. A threshold policy should only be recommended if a counterfactual threshold experiment performs similarly to the flexible rollout policy.

## Optional Counterfactual Analysis Template

Use this section only after generating additional runs. The baseline notebook is intentionally centered on the integrated rollout run; these cells are for quantifying losses or marginal effects once explicit comparison folders exist.

In [ ]:
# Optional: compare the primary baseline to additional run folders listed in OPTIONAL_COMPARISON_RUN_DIRS.
if not comparisons["results"].empty:
    comparison_results = pd.concat([results, comparisons["results"]], ignore_index=True, sort=False)
    comparison_summary = mean_ci(comparison_results, ["policy_label"], "service_level").sort_values("mean", ascending=False)
    display(comparison_summary)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(comparison_summary["policy_label"], comparison_summary["mean"], yerr=comparison_summary["ci95"], capsize=4)
    ax.set_title("Service Level Across Baseline and Counterfactual Runs")
    ax.set_ylabel("Service level")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
else:
    print("No optional comparison folders configured. Add folders to OPTIONAL_COMPARISON_RUN_DIRS to quantify integration gain, repair-time thresholds, or vehicle-count marginal benefits.")

## Recommended Result Schema Additions

The current run already supports many behavioral insights. The following additions would make the managerial chapter stronger and reduce path-based inference:

- `run_id` in every CSV file.
- `policy_name`, `restriction_type`, and `restriction_active` in `decisions.csv`.
- `vehicle_id` in `decisions.csv`, plus active, idle, travel, repair, and depot time by vehicle.
- `go_to_depot` as a selected-routing flag, in addition to `next_station_id`.
- `vehicle_capacity` explicitly logged at each decision.
- `repair_time` and `num_service_vehicles` explicitly logged in `results.csv`.
- Candidate diagnostics: number of maintenance candidates available, number removed by restrictions, and whether the restriction changed the selected action.